# 3. Parallel & Conditional Chains
**Industry:** Insurance

Analyze a claim's damage and fraud risk in parallel, then route to auto-approve or manual review.

In [ ]:
!pip install langchain langchain-google-genai python-dotenv

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

damage_prompt = ChatPromptTemplate.from_template("Assess the damage severity (Low/Medium/High) for: {claim}")
fraud_prompt = ChatPromptTemplate.from_template("Assess fraud risk (Score 1-10) for: {claim}. Only return the number.")

parallel_chain = RunnableParallel(
    damage=damage_prompt | llm | StrOutputParser(),
    fraud_score=fraud_prompt | llm | StrOutputParser()
)

def route_claim(data):
    try:
        score = int(data['fraud_score'].strip())
    except:
        score = 10
    if score < 4 and 'High' not in data['damage']:
        return "Auto-Approve: Claim seems legitimate and damage is not severe."
    else:
        return "Manual Review Required: High fraud risk or severe damage detected."

full_chain = parallel_chain | RunnableLambda(route_claim)

claim_desc = "Car was rear-ended at a stop light. Bumper is dented."
print(full_chain.invoke({"claim": claim_desc}))